In [1]:
import pandas as pd
import os


In [2]:
df = pd.read_csv("experiment_log.csv")
columns = {
    "deepseek-r1:latest": "DeepSeek-R1",
    "llama3.2:latest": "Llama3.2",
    "qwen2.5:latest": "Qwen2.5",
    "qwen3.5:latest": "Qwen3.5",
    "mistral": "Mistral",
    "assist2009": "ASSISTments 2009",
}
df["dataset_name"] = df["dataset_name"].apply(lambda x: columns[x])
df.rename(columns={"model_name": "Model", "dataset_name": "Dataset"}, inplace=True)
df

,Dataset,Model,trainauc,trainacc,validauc,validacc,testauc,testacc
0,Llama3.2,dkt,0.894802,0.868771,0.869210,0.855104,0.871442,0.854701
1,Mistral,dkt,0.884595,0.860516,0.876764,0.857985,0.867343,0.865052
2,Qwen3.5,dkt,0.888498,0.861953,0.895596,0.866511,0.869190,0.859907
3,Qwen2.5,dkt,0.882718,0.860844,0.860292,0.870810,0.876316,0.857752
4,DeepSeek-R1,dkt,0.885416,0.867389,0.897946,0.867511,0.872925,0.865039
5,ASSISTments 2009,dkt,0.865165,0.826422,0.798089,0.784864,0.794926,0.782129
6,Llama3.2,sakt,0.910440,0.882080,0.872203,0.862788,0.881908,0.864103
7,Mistral,sakt,0.898530,0.868651,0.888369,0.850913,0.871033,0.843714
8,Qwen3.5,sakt,0.900426,0.868013,0.891581,0.866511,0.882633,0.859239
9,Qwen2.5,sakt,0.900130,0.862837,0.870808,0.868017,0.880079,0.855619


In [3]:
df.sort_values(by=["Model", "Dataset"], inplace=True, ignore_index=True)
df

,Dataset,Model,trainauc,trainacc,validauc,validacc,testauc,testacc
0,ASSISTments 2009,akt,0.852197,0.813949,0.790921,0.784635,0.785136,0.782852
1,DeepSeek-R1,akt,0.873320,0.863711,0.896214,0.860759,0.877138,0.859254
2,Llama3.2,akt,0.883193,0.845559,0.874985,0.863886,0.867517,0.861538
3,Mistral,akt,0.879509,0.840675,0.884417,0.858574,0.870154,0.865052
4,Qwen2.5,akt,0.877178,0.853415,0.862359,0.844274,0.869934,0.835704
5,Qwen3.5,akt,0.886606,0.846689,0.886112,0.866511,0.874426,0.859907
6,ASSISTments 2009,dkt,0.865165,0.826422,0.798089,0.784864,0.794926,0.782129
7,DeepSeek-R1,dkt,0.885416,0.867389,0.897946,0.867511,0.872925,0.865039
8,Llama3.2,dkt,0.894802,0.868771,0.869210,0.855104,0.871442,0.854701
9,Mistral,dkt,0.884595,0.860516,0.876764,0.857985,0.867343,0.865052


In [4]:
def create_pivot_table(
    df: pd.DataFrame, value_column: str, table_name: str
) -> pd.DataFrame:
    pt = pd.pivot_table(
        df,
        values=value_column,
        index="Dataset",
        columns="Model",
        margins=True,
        margins_name="Average",
    )

    column_format = "+l^c^c^c^c^c^>{\\bfseries}^c"
    caption = f"{table_name} across LLM Synthesizers and Knowledge Tracing Models."
    label = f"\\label{{tab:{value_column}}}"

    pt_latex: str = pt.to_latex(
        float_format="%.4f",
        column_format=column_format,
        index_names=False,
        header=["AKT", "DKT", "DTransformer", "SAKT", "SimpleKT", "Average"],
        caption=caption + label,
        position="H",
    )
    kt_title = "&\\multicolumn{6}{c}{\\textbf{Knowledge Tracing Algorithms}}\\\\ \n"
    bold_rowstyle = "\\rowstyle{\\bfseries}"

    pt_latex = pt_latex.replace("\nModel", "\nSynthesizer")
    pt_latex = pt_latex.replace("[H]", "[H]\n\\centering")
    pt_latex = pt_latex.replace("DeepSeek-R1", "\\midrule\nDeepSeek-R1")
    pt_latex = pt_latex.replace("\\toprule", "\\toprule\n" + kt_title + bold_rowstyle)
    pt_latex = pt_latex.replace("\nAverage", f"\n\\midrule\n{bold_rowstyle}\nAverage")

    os.makedirs("tables", exist_ok=True)
    with open(f"../dissertation/tables/{value_column}.tex", "w") as f:
        f.write(pt_latex)

    return pt

In [5]:
pivot_table_names = {
    "trainacc": "Last epoch \\textbf{Train Accuracy}",
    "trainauc": "Last epoch \\textbf{Train AUC}",
    "validacc": "Last epoch \\textbf{Validation Accuracy}",
    "validauc": "Last epoch \\textbf{Validation AUC}",
    "testacc": "\\textbf{Test Accuracy}",
    "testauc": "\\textbf{Test AUC}",
}

tables = {}
for value, table_name in pivot_table_names.items():
    tables[value] = create_pivot_table(df, value, table_name)

In [6]:
print(abs(tables["trainauc"] - tables["validauc"]).mean().mean())
print(abs(tables["trainauc"] - tables["testauc"]).mean().mean())
print(abs(tables["trainacc"] - tables["validacc"]).mean().mean())
print(abs(tables["trainacc"] - tables["testacc"]).mean().mean())

print()

print(abs(tables["trainauc"] - tables["validauc"]).T.mean().mean())
print(abs(tables["trainauc"] - tables["testauc"]).T.mean().mean())
print(abs(tables["trainacc"] - tables["validacc"]).T.mean().mean())
print(abs(tables["trainacc"] - tables["testacc"]).T.mean().mean())

0.03165043501194414
0.032577758242528024
0.018702932919535526
0.01884478714276637

0.03165043501194414
0.032577758242528024
0.018702932919535526
0.018844787142766373


In [7]:
pd.set_option("display.width", 88)
pd.set_option("display.max_columns", None)

print("trainauc - validauc")
print(abs(tables["trainauc"] - tables["validauc"]))
print()

print("trainauc - testauc")
print(abs(tables["trainauc"] - tables["testauc"]))
print()

print("trainacc - validacc")
print(abs(tables["trainacc"] - tables["validacc"]))
print()

print("trainacc - testacc")
print(abs(tables["trainacc"] - tables["testacc"]))


trainauc - validauc
Model                  akt       dkt  dtransformer      sakt  simplekt   Average
Dataset                                                                         
ASSISTments 2009  0.061276  0.067076      0.161428  0.189488  0.162182  0.128290
DeepSeek-R1       0.022894  0.012530      0.006724  0.005279  0.011016  0.008999
Llama3.2          0.008208  0.025592      0.009170  0.038237  0.039847  0.024211
Mistral           0.004908  0.007831      0.001954  0.010161  0.007490  0.004506
Qwen2.5           0.014818  0.022426      0.004645  0.029323  0.021802  0.018603
Qwen3.5           0.000493  0.007098      0.005575  0.008845  0.004754  0.002514
Average           0.009499  0.017216      0.031583  0.045129  0.037510  0.028187

trainauc - testauc
Model                  akt       dkt  dtransformer      sakt  simplekt   Average
Dataset                                                                         
ASSISTments 2009  0.067061  0.070239      0.160277  0.176656  0.16625

In [8]:
print(abs(tables["testacc"] - tables["trainacc"]).T.mean())


Dataset
ASSISTments 2009    0.067489
DeepSeek-R1         0.005206
Llama3.2            0.019638
Mistral             0.012172
Qwen2.5             0.006256
Qwen3.5             0.006355
Average             0.014797
dtype: float64
